## Paso 1 — Cargamos las 18 tablas ya tipadas

Leemos directo los Parquet generados en `00_setup_y_carga.ipynb` (ya sin
fila fantasma, con piso-sicetac separado en núcleo/raras, tipados y en
camelCase) — sin depender de que ese notebook se haya corrido antes en esta
misma sesión. Esto es lo único que este notebook necesita del anterior.


In [1]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))

tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}

assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"
print(f"Tablas cargadas: {len(tablas)}")
for nombre, df in tablas.items():
    print(f"  {nombre}: {df.shape[0]:>6,} filas x {df.shape[1]:>2} columnas")


Tablas cargadas: 18
  audit_log:     72 filas x  6 columnas
  catalogo-clientes-BUT000-organizaciones: 38,995 filas x 10 columnas
  catalogo_clientes:     21 filas x  6 columnas
  catalogo_configuraciones_vehiculo:     16 filas x  6 columnas
  catalogo_rutas:     31 filas x  5 columnas
  catalogo_servicios_conexos:     11 filas x  8 columnas
  catalogo_usuarios:     11 filas x  4 columnas
  cotizacion_conexos:      7 filas x  5 columnas
  cotizacion_lineas:     48 filas x  9 columnas
  cotizaciones:     18 filas x  9 columnas
  historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy: 25,318 filas x 20 columnas
  inconsistencias:      1 filas x  3 columnas
  lineas_solicitud:     43 filas x 18 columnas
  piso-sicetac_nucleo:    126 filas x 16 columnas
  piso-sicetac_raras:     25 filas x 16 columnas
  solicitudes:     13 filas x 12 columnas
  tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy: 23,020 filas x 24 columnas
  tarifas_historicas:     15 filas x  7 columnas


## Paso 2 — Dimensiones y peso en memoria

Filas, columnas y memoria RAM real que ocupa cada una de las 18 tablas ya
cargadas — para tener una idea de tamaño relativo antes de perfilar columna
por columna.


In [2]:
resumen_tablas = []
for nombre, df in tablas.items():
    memoria_kb = df.memory_usage(deep=True).sum() / 1024
    resumen_tablas.append({
        "tabla": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "memoria_kb": round(memoria_kb, 1),
    })

df_resumen_tablas = pd.DataFrame(resumen_tablas).sort_values("memoria_kb", ascending=False).reset_index(drop=True)
df_resumen_tablas


,tabla,filas,columnas,memoria_kb
0,tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy,23020,24,7153.1
1,historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy,25318,20,5991.4
2,catalogo-clientes-BUT000-organizaciones,38995,10,5085.1
3,piso-sicetac_nucleo,126,16,28.1
4,audit_log,72,6,12.7
5,lineas_solicitud,43,18,10.4
6,cotizacion_lineas,48,9,9.0
7,piso-sicetac_raras,25,16,6.4
8,cotizaciones,18,9,3.2
9,catalogo_rutas,31,5,3.1


## Paso 3 — Perfil columna por columna (18 tablas)

Por cada columna: tipo real, nulos, % de nulos, valores únicos, cardinalidad
relativa (únicos / filas — cerca de 1 sugiere un identificador; cerca de 0
sugiere una categoría con pocas opciones), el valor más frecuente y qué
porcentaje de las filas lo tienen (para detectar columnas cuasi-constantes,
no solo constantes puras), y un ejemplo real.


In [3]:
filas_perfil = []
for nombre, df in tablas.items():
    n = len(df)
    for columna in df.columns:
        serie = df[columna]
        no_nulos = serie.dropna()
        valor_frecuente = no_nulos.mode().iloc[0] if not no_nulos.empty else None
        pct_valor_frecuente = (no_nulos == valor_frecuente).mean() * 100 if not no_nulos.empty else 0.0
        filas_perfil.append({
            "tabla": nombre,
            "columna": columna,
            "dtype": str(serie.dtype),
            "nulos": int(serie.isna().sum()),
            "pct_nulos": round(serie.isna().mean() * 100, 1),
            "unicos": int(no_nulos.nunique()),
            "cardinalidad_relativa": round(no_nulos.nunique() / n, 3) if n else 0.0,
            "valor_mas_frecuente": valor_frecuente,
            "pct_valor_mas_frecuente": round(pct_valor_frecuente, 1),
            "ejemplo": no_nulos.iloc[0] if not no_nulos.empty else None,
        })

df_perfil_columnas = pd.DataFrame(filas_perfil)
print(f"Columnas perfiladas: {len(df_perfil_columnas)}")
df_perfil_columnas


Columnas perfiladas: 184


,tabla,columna,dtype,nulos,pct_nulos,unicos,cardinalidad_relativa,valor_mas_frecuente,pct_valor_mas_frecuente,ejemplo
0,audit_log,id,str,0,0.0,72,1.000,03930948-029C-4B56-A620-54FE98B613FC,1.4,50A6E1BF-E7F8-42D8-9D40-002FA749B109
1,audit_log,usuarioId,str,0,0.0,1,0.014,00000000-0000-0000-0000-000000000001,100.0,00000000-0000-0000-0000-000000000001
2,audit_log,entidad,str,0,0.0,4,0.056,cotizaciones,59.7,cotizaciones
3,audit_log,entidadId,str,0,0.0,39,0.542,0EE0943A-81F2-49F0-A6B8-AF5B2B04FA51,11.1,D782291B-35C9-4532-B845-4A765C1CFF3C
4,audit_log,accion,str,0,0.0,12,0.167,crear,23.6,entregar
...,...,...,...,...,...,...,...,...,...,...
179,tarifas_historicas,configVehiculoId,str,11,73.3,4,0.267,03D0965A-74D8-49FE-8C0E-9861F6306E0A,25.0,8D185987-45FD-409F-B656-D432E77EED28
180,tarifas_historicas,tipologia,str,12,80.0,3,0.200,CargaSeca,33.3,GranelSolido
181,tarifas_historicas,valor,float64,0,0.0,13,0.867,1900000.0,20.0,6788888.0
182,tarifas_historicas,fecha,datetime64[us],0,0.0,2,0.133,2026-08-31 00:00:00,80.0,2026-08-31 00:00:00


## Paso 4 — Columnas constantes o cuasi-constantes

Filtramos del perfil anterior las columnas donde un solo valor domina 95%
o más de las filas — son columnas que, tal como están en este set, aportan
poca o ninguna señal para diferenciar un registro de otro. No se descartan
solo por esto (puede ser un hecho real y esperado, como `TYPE` siempre
`'2'` en el catálogo de clientes SAP, ya documentado) — se listan para
decidir caso por caso, no en automático.


In [4]:
columnas_cuasi_constantes = df_perfil_columnas[
    df_perfil_columnas["pct_valor_mas_frecuente"] >= 95
].sort_values("pct_valor_mas_frecuente", ascending=False)

print(f"Columnas cuasi-constantes (>=95% un solo valor): {len(columnas_cuasi_constantes)} de {len(df_perfil_columnas)}")
columnas_cuasi_constantes[["tabla", "columna", "unicos", "valor_mas_frecuente", "pct_valor_mas_frecuente"]]


Columnas cuasi-constantes (>=95% un solo valor): 25 de 184


,tabla,columna,unicos,valor_mas_frecuente,pct_valor_mas_frecuente
1,audit_log,usuarioId,1,00000000-0000-0000-0000-000000000001,100.0
7,catalogo-clientes-BUT000-organizaciones,type,1,2,100.0
15,catalogo-clientes-BUT000-organizaciones,validTo,1,9999-12-31 23:59:59,100.0
46,cotizacion_conexos,cotizacionId,1,0EE0943A-81F2-49F0-A6B8-AF5B2B04FA51,100.0
64,cotizaciones,entregadaPor,1,00000000-0000-0000-0000-000000000001,100.0
88,inconsistencias,entidad,1,solicitudes,100.0
89,inconsistencias,entidadId,1,3EAB0AB6-ED6D-4983-A8EA-F9BF7406AB0A,100.0
90,inconsistencias,motivo,1,Deadline anterior a la fecha de creacion,100.0
95,lineas_solicitud,tipologia,1,CargaSeca,100.0
102,lineas_solicitud,levanteContenedorDiferente,1,False,100.0


In [5]:
columnas_cuasi_constantes = df_perfil_columnas[
    df_perfil_columnas["pct_valor_mas_frecuente"] >= 95
].sort_values("pct_valor_mas_frecuente", ascending=False)

print(f"Columnas cuasi-constantes (>=95% un solo valor): {len(columnas_cuasi_constantes)} de {len(df_perfil_columnas)}")
columnas_cuasi_constantes[["tabla", "columna", "nulos", "pct_nulos", "unicos", "valor_mas_frecuente", "pct_valor_mas_frecuente"]]


Columnas cuasi-constantes (>=95% un solo valor): 25 de 184


,tabla,columna,nulos,pct_nulos,unicos,valor_mas_frecuente,pct_valor_mas_frecuente
1,audit_log,usuarioId,0,0.0,1,00000000-0000-0000-0000-000000000001,100.0
7,catalogo-clientes-BUT000-organizaciones,type,0,0.0,1,2,100.0
15,catalogo-clientes-BUT000-organizaciones,validTo,18,0.0,1,9999-12-31 23:59:59,100.0
46,cotizacion_conexos,cotizacionId,0,0.0,1,0EE0943A-81F2-49F0-A6B8-AF5B2B04FA51,100.0
64,cotizaciones,entregadaPor,12,66.7,1,00000000-0000-0000-0000-000000000001,100.0
88,inconsistencias,entidad,0,0.0,1,solicitudes,100.0
89,inconsistencias,entidadId,0,0.0,1,3EAB0AB6-ED6D-4983-A8EA-F9BF7406AB0A,100.0
90,inconsistencias,motivo,0,0.0,1,Deadline anterior a la fecha de creacion,100.0
95,lineas_solicitud,tipologia,42,97.7,1,CargaSeca,100.0
102,lineas_solicitud,levanteContenedorDiferente,0,0.0,1,False,100.0


## Paso 5 — Cobertura de las llaves de cruce (§3.2 del plan)

Para cada llave que conecta lo propio con lo externo: qué % de las filas la
trae poblada, y de esas, qué % encuentra un valor igual en la tabla
destino. Este número determina qué tan grande puede ser cualquier análisis
de "señal de mercado" en la Fase 6 — si la cobertura real es baja, hay que
ajustar la expectativa del entregable desde ya, no descubrirlo al final.
La llave de Ruta (texto libre) se deja fuera de este cálculo a propósito —
normalizarla es trabajo de la Fase 8, no de aquí.


In [6]:
def cobertura_llave(df_origen, col_origen, valores_destino):
    poblado = df_origen[col_origen].notna()
    pct_poblado = poblado.mean() * 100
    valores_poblados = df_origen.loc[poblado, col_origen]
    hace_match = valores_poblados.isin(valores_destino)
    n_poblados = len(valores_poblados)
    pct_match = hace_match.mean() * 100 if n_poblados else float("nan")
    return pct_poblado, pct_match, n_poblados, int(hace_match.sum())

tabla_lineas_solicitud = tablas["lineas_solicitud"]
tabla_tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]
tabla_catalogo_clientes = tablas["catalogo_clientes"]
tabla_but000 = tablas["catalogo-clientes-BUT000-organizaciones"]

filas_cobertura = []

for columna_propio, columna_externo in [
    ("lineaNegocio", "lineanegocio"),
    ("claseViaje", "claseviaje"),
    ("claseContenedor", "clasecontenedor"),
]:
    valores_destino = set(tabla_tarifa_plaza[columna_externo].dropna().unique())
    pct_poblado, pct_match, n_poblados, n_match = cobertura_llave(tabla_lineas_solicitud, columna_propio, valores_destino)
    filas_cobertura.append({
        "llave": columna_propio,
        "tabla_origen": "lineas_solicitud",
        "tabla_destino": "tarifa-plaza",
        "pct_poblado": round(pct_poblado, 1),
        "filas_pobladas": n_poblados,
        "filas_con_match": n_match,
        "pct_match_de_pobladas": round(pct_match, 1) if n_poblados else None,
    })

valores_nit_destino = set(tabla_but000["taxnum"].dropna().unique())
pct_poblado, pct_match, n_poblados, n_match = cobertura_llave(tabla_catalogo_clientes, "nit", valores_nit_destino)
filas_cobertura.append({
    "llave": "nit",
    "tabla_origen": "catalogo_clientes",
    "tabla_destino": "catalogo-clientes-BUT000 (taxnum)",
    "pct_poblado": round(pct_poblado, 1),
    "filas_pobladas": n_poblados,
    "filas_con_match": n_match,
    "pct_match_de_pobladas": round(pct_match, 1) if n_poblados else None,
})

df_cobertura_llaves = pd.DataFrame(filas_cobertura)
df_cobertura_llaves


,llave,tabla_origen,tabla_destino,pct_poblado,filas_pobladas,filas_con_match,pct_match_de_pobladas
0,lineaNegocio,lineas_solicitud,tarifa-plaza,9.3,4,0,0.0
1,claseViaje,lineas_solicitud,tarifa-plaza,11.6,5,0,0.0
2,claseContenedor,lineas_solicitud,tarifa-plaza,4.7,2,0,0.0
3,nit,catalogo_clientes,catalogo-clientes-BUT000 (taxnum),100.0,21,1,4.8


In [7]:
MAPEO_LINEA_NEGOCIO = {
    "Andino": "7", "DevolucionContenedor": "6", "Nacional": "4",
    "PuertoAInterior": "1", "InteriorAPuerto": "2", "Licitaciones": "8",
}
MAPEO_CLASE_VIAJE = {
    "Contenedor": "1", "Cupo": "2", "CargaSuelta": "4", "Extradimensionado": "5",
}

def cobertura_llave_traducida(df_origen, col_origen, mapeo, valores_destino):
    poblado = df_origen[col_origen].notna()
    n_poblados = int(poblado.sum())
    valores_traducidos = df_origen.loc[poblado, col_origen].map(mapeo)
    sin_mapeo_conocido = int(valores_traducidos.isna().sum())
    hace_match = valores_traducidos.isin(valores_destino)
    n_match = int(hace_match.sum())
    pct_poblado = poblado.mean() * 100
    pct_match = (n_match / n_poblados * 100) if n_poblados else float("nan")
    return pct_poblado, pct_match, n_poblados, n_match, sin_mapeo_conocido

valores_destino_lineanegocio = set(tabla_tarifa_plaza["lineanegocio"].dropna().unique())
valores_destino_claseviaje = set(tabla_tarifa_plaza["claseviaje"].dropna().unique())

for nombre_llave, columna, mapeo, valores_destino in [
    ("lineaNegocio (traducido)", "lineaNegocio", MAPEO_LINEA_NEGOCIO, valores_destino_lineanegocio),
    ("claseViaje (traducido)", "claseViaje", MAPEO_CLASE_VIAJE, valores_destino_claseviaje),
]:
    pct_poblado, pct_match, n_poblados, n_match, sin_mapeo = cobertura_llave_traducida(
        tabla_lineas_solicitud, columna, mapeo, valores_destino
    )
    print(f"{nombre_llave}: poblado {pct_poblado:.1f}% ({n_poblados} filas) | "
          f"match tras traducir: {n_match}/{n_poblados} ({pct_match:.1f}%) | "
          f"valores sin traducción conocida: {sin_mapeo}")

print("\nclasecontenedor: NO verificable con este extracto — requiere la tabla "
      "tblSAPClaseContenedor (catálogo del backend), que no forma parte de las "
      "4 fuentes externas entregadas en v1.0.0-piloto.")


lineaNegocio (traducido): poblado 9.3% (4 filas) | match tras traducir: 1/4 (25.0%) | valores sin traducción conocida: 0
claseViaje (traducido): poblado 11.6% (5 filas) | match tras traducir: 5/5 (100.0%) | valores sin traducción conocida: 0

clasecontenedor: NO verificable con este extracto — requiere la tabla tblSAPClaseContenedor (catálogo del backend), que no forma parte de las 4 fuentes externas entregadas en v1.0.0-piloto.


## Paso 6 — Chequeo preliminar de Ruta (adelantado de la Fase 8, por prioridad de negocio)

El motor real de costeo filtra principalmente por ruta, no por los códigos
de LineaNegocio/ClaseViaje — así que esta llave es la más importante de
todo el cruce, no una más. Este chequeo es **preliminar**: limpieza simple
del nombre de ciudad + comparación exacta, no la normalización fina
(fuzzy matching) que le corresponde a la Fase 8. El objetivo aquí es solo
saber si hay alguna esperanza de conectar, antes de invertir el trabajo
fino más adelante.


In [8]:
import re
import unicodedata

def limpiar_nombre_ciudad(texto):
    if pd.isna(texto):
        return None
    sin_departamento = re.sub(r"\s*\([^)]*\)\s*$", "", texto).strip()
    sin_tildes = unicodedata.normalize("NFKD", sin_departamento).encode("ascii", "ignore").decode("ascii")
    return sin_tildes.upper()

tabla_catalogo_rutas = tablas["catalogo_rutas"]
tabla_sicetac_nucleo = tablas["piso-sicetac_nucleo"]
tabla_sicetac_raras = tablas["piso-sicetac_raras"]
tabla_historico_sap = tablas["historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy"]

# Excluir las 2 filas ya documentadas como basura de prueba (reporte-inconsistencias.md)
rutas_validas = tabla_catalogo_rutas[
    ~tabla_catalogo_rutas["origen"].isin(["222", "b"])
].copy()
print(f"catalogo_rutas: {len(tabla_catalogo_rutas)} filas totales, {len(rutas_validas)} tras excluir basura de prueba ya documentada")

rutas_validas["origen_limpio"] = rutas_validas["origen"].apply(limpiar_nombre_ciudad)
rutas_validas["destino_limpio"] = rutas_validas["destino"].apply(limpiar_nombre_ciudad)

valores_sap_completo = (
    set(tabla_tarifa_plaza["origen"].dropna().unique())
    | set(tabla_tarifa_plaza["destino"].dropna().unique())
    | set(tabla_historico_sap["origen"].dropna().unique())
    | set(tabla_historico_sap["destino"].dropna().unique())
)
texto_sicetac_completo = " | ".join(
    tabla_sicetac_nucleo["nombreruta"].dropna().tolist()
    + tabla_sicetac_raras["nombreruta"].dropna().tolist()
)

def hace_match_sap(ciudad_limpia):
    if ciudad_limpia is None:
        return False
    return f"ZC_{ciudad_limpia}" in valores_sap_completo or f"ZTC_{ciudad_limpia}" in valores_sap_completo

def hace_match_sicetac(ciudad_limpia):
    if ciudad_limpia is None:
        return False
    return ciudad_limpia in texto_sicetac_completo

rutas_validas["origen_match_sap"] = rutas_validas["origen_limpio"].apply(hace_match_sap)
rutas_validas["destino_match_sap"] = rutas_validas["destino_limpio"].apply(hace_match_sap)
rutas_validas["origen_match_sicetac"] = rutas_validas["origen_limpio"].apply(hace_match_sicetac)
rutas_validas["destino_match_sicetac"] = rutas_validas["destino_limpio"].apply(hace_match_sicetac)

print(f"\nMatch contra SAP completo (tarifa-plaza + histórico, ZC_ o ZTC_):")
print(f"  origen: {rutas_validas['origen_match_sap'].mean()*100:.1f}%  |  destino: {rutas_validas['destino_match_sap'].mean()*100:.1f}%")
print(f"Match contra SICE-TAC completo (núcleo + raras):")
print(f"  origen: {rutas_validas['origen_match_sicetac'].mean()*100:.1f}%  |  destino: {rutas_validas['destino_match_sicetac'].mean()*100:.1f}%")

rutas_validas[["origen", "destino", "origen_limpio", "destino_limpio", "origen_match_sap", "destino_match_sap", "origen_match_sicetac", "destino_match_sicetac"]]
   

catalogo_rutas: 31 filas totales, 29 tras excluir basura de prueba ya documentada

Match contra SAP completo (tarifa-plaza + histórico, ZC_ o ZTC_):
  origen: 86.2%  |  destino: 69.0%
Match contra SICE-TAC completo (núcleo + raras):
  origen: 86.2%  |  destino: 75.9%


,origen,destino,origen_limpio,destino_limpio,origen_match_sap,destino_match_sap,origen_match_sicetac,destino_match_sicetac
0,Medellín (ANT),Guarne (ANT),MEDELLIN,GUARNE,True,True,True,True
1,Cartagena,Bogotá,CARTAGENA,BOGOTA,True,True,True,True
2,barranquilla,medellin,BARRANQUILLA,MEDELLIN,True,True,True,True
3,Medellin,Bogota,MEDELLIN,BOGOTA,True,True,True,True
4,Cali,Bogotá,CALI,BOGOTA,True,True,True,True
7,Bogotá,Medellín,BOGOTA,MEDELLIN,True,True,True,True
8,Dosquebradas (RIS),Rionegro (ANT),DOSQUEBRADAS,RIONEGRO,True,False,True,False
9,Cali,Buenaventura,CALI,BUENAVENTURA,True,True,True,True
10,Barranquilla,Bogotá,BARRANQUILLA,BOGOTA,True,True,True,True
11,Funza (CUN),Manizales (CAL),FUNZA,MANIZALES,True,True,True,True


## Síntesis del notebook 01 — Perfilado estructural (Fase 2, §3 del plan)

**Objetivo cumplido:** inventario estructural de las 18 tablas y, sobre todo,
la cobertura real de las llaves de cruce — el entregable central de esta
fase según el plan.

**Perfilado por tabla (Pasos 1-3):** 18 tablas cargadas desde los Parquet
del notebook 00, todas con las filas documentadas. 184 columnas perfiladas
(tipo, nulos, cardinalidad, valor más frecuente). Detectamos y corregimos
en el camino un bug real del notebook 00: `dayfirst=True` aplicado de forma
global rompía fechas ISO del lado propio (ver síntesis del notebook 00) —
se encontró precisamente porque una fecha corrupta aparecía como
"cuasi-constante" en este perfilado.

**Columnas cuasi-constantes (Paso 4):** 25 de 184 columnas con un solo
valor en ≥95% de las filas (tras corregir el bug de fechas). La mayoría es
esperable a esta escala piloto (un solo usuario de prueba, tablas de 1-18
filas) o ya estaba documentado (`TYPE='2'` en clientes SAP). Dos candidatas
reales para `hallazgos.md`:
- `catalogo-clientes-BUT000.validTo` = 100% el valor centinela
  `9999-12-31` en las 38.995 filas, sin una sola excepción — no aporta
  señal alguna en este extracto.
- `solicitudes.modalidad/probabilidadCierre/valorEstimado` (92,3% vacíos) y
  `lineas_solicitud.tipologia` (97,7% vacíos) — confirmado por el usuario
  como hueco conocido de la fase piloto de CabinaFletes, no un bug de este
  EDA; se documenta como limitación heredada, no como hallazgo a resolver.

**Cobertura de llaves de negocio — `LineaNegocio`/`ClaseViaje`/
`ClaseContenedor` (Paso 5):** muy baja población (4,7%-11,6% de
`lineas_solicitud`) y, comparando texto literal contra `tarifa-plaza`, 0%
de match. Investigación en el código del backend
(`TarifarioSapAdapter.cs`) confirmó que **sí existe traducción**, pero vive
en diccionarios fijos de C#, no en ninguna tabla de este extracto — al
traducir, `claseviaje` sube a 100% de match (5/5) y `lineanegocio` a 25%
(1/4). `ClaseContenedor` no tiene mapeo fijo: se resuelve contra una tabla
externa (`tblSAPClaseContenedor`) ausente de las 4 fuentes entregadas —
no verificable con este extracto. Corrige una afirmación del diccionario
`diccionario-y-linaje.md` (HU-10.3) de que estas columnas son "la misma
llave" — en la práctica requieren traducción, y una de las tres ni siquiera
es verificable aquí.

**Corrección de prioridad de negocio, confirmada por el usuario:** el motor
real de costeo filtra principalmente por **ruta**, no por estos códigos —
son secundarios. Esto reordena qué tan grave es la baja cobertura anterior:
no es la pieza que más importa.

**Cobertura de Ruta — chequeo preliminar adelantado de la Fase 8 (Paso 6):**
con limpieza mínima (quitar departamento entre paréntesis, quitar tildes),
el match subió de 51,7%/34,5% a **86,2% origen / 69-76% destino** contra
SAP y SICE-TAC combinados — mucho más prometedor que las llaves de negocio.
Hallazgo de mecanismo: SAP no siempre usa código individual por ciudad
(`ZC_<ciudad>`) — para zonas metropolitanas grandes usa código de **zona de
tarificación** (`ZTC_BOGOTA`, `ZTC_VABURRA`), consistente con el hallazgo ya
conocido de `tblZonasTarificacionRelacion`. Fallos restantes identificados
como candidatos concretos para la Fase 8: sufijo "D.C." sin limpiar
(`"Bogotá D.C."` no calza aunque `"Bogotá"` sola sí) y nombre formal vs.
común (`"Santiago de Cali"` vs. `"Cali"`).

**Conclusión de la fase:** de las 4 llaves evaluadas, Ruta es la única con
señal real de conexión a esta escala. Las llaves de negocio, aunque
traducibles, tienen población demasiado baja para aportar mucho. NIT
quedó con 100% de población pero solo 4,8% de match (1 de 21) — coherente
con que el catálogo de clientes propio es dato de prueba manual, no
necesariamente NITs reales de las mismas empresas que SAP tiene
registradas.

**Qué le queda listo al siguiente notebook:** el entendimiento de qué
llave usar (Ruta) y cuál no priorizar (códigos de negocio) para cuando
llegue la Fase 6 (bivariado). `02_unicidad_y_granularidad.ipynb` (Fase 3)
parte de aquí, sin depender de este perfilado salvo para contexto.
